In [ ]:
# ── 01: build the pollution panel ──────────────────────────────────
!pip install netcdf4 xarray -q

from google.colab import drive
drive.mount('/content/drive')

import glob, numpy as np, pandas as pd
DIR = "/content/drive/MyDrive/pollution-ah"
print(sorted(f.split("/")[-1] for f in glob.glob(f"{DIR}/*")))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['01_build_pollution.ipynb', 'hk_pm25_2013_2017.csv', 'hk_pm25_2018_2022.csv', 'hk_pm25_2023_2026.csv']


In [ ]:
# ── Hong Kong daily PM2.5 ──────────────────────────────────────────
# 15 general stations + 3 roadside (Central, Mong Kok, Causeway Bay).
# Roadside read systematically higher and are excluded from the city
# mean, kept only as a robustness check. North and Southern have ~40%
# coverage and are dropped so the station set stays stable over time.

GENERAL = ["SHATIN","TSUEN WAN","EASTERN","KWUN TONG","TUEN MUN","TUNG CHUNG",
           "SHAM SHUI PO","YUEN LONG","CENTRAL/WESTERN","KWAI CHUNG",
           "TAP MUN","TSEUNG KWAN O","TAI PO"]
ROADSIDE = ["CENTRAL","MONG KOK","CAUSEWAY BAY"]
MIN_STATIONS = 8

parts = []
for f in sorted(glob.glob(f"{DIR}/hk_pm25_*.csv")):
    d = pd.read_csv(f, skiprows=11, na_values=["N.A."], encoding="utf-8-sig")
    d.columns = [c.strip() for c in d.columns]
    parts.append(d)
    print(f"  {f.split('/')[-1]}: {len(d)} rows")

H = pd.concat(parts, ignore_index=True)
H["ts"] = (pd.to_datetime(H.DATE, format="%d/%m/%Y")
           + pd.to_timedelta(H.HOUR.astype(int) - 1, unit="h"))   # HOUR is 1–24
H = H.drop_duplicates("ts").set_index("ts").sort_index()

gen = H[GENERAL].apply(pd.to_numeric, errors="coerce")
n_st = gen.notna().sum(axis=1)
hk_hourly = gen.mean(axis=1).where(n_st >= MIN_STATIONS)

hk = hk_hourly.resample("D").mean().dropna()
hk.name = "hk_pm25"

print(f"\nhourly: {len(H)} rows, {H.index.min().date()} to {H.index.max().date()}")
print(f"stations used: {len(GENERAL)} general (North, Southern dropped for ~40% coverage)")
print(f"daily series: {len(hk)} days")
print(f"  mean {hk.mean():.1f}   sd {hk.std():.1f}   min {hk.min():.1f}   max {hk.max():.1f}")

  hk_pm25_2013_2017.csv: 43824 rows
  hk_pm25_2018_2022.csv: 43824 rows
  hk_pm25_2023_2026.csv: 28464 rows

hourly: 116112 rows, 2013-01-01 to 2026-03-31
stations used: 13 general (North, Southern dropped for ~40% coverage)
daily series: 4838 days
  mean 19.6   sd 13.1   min 2.4   max 118.0


In [ ]:
# ── mainland China daily PM2.5, all cities ─────────────────────────
# Raw MEP station observations from Zenodo 10.5281/zenodo.6404603
# (Li & Li, Institute of Earth Environment, CAS; CC BY 4.0).
# Winters only: Dec/Jan/Feb, 2013-14 through 2019-20.
#
# Cleaning, all discovered by inspection rather than documented:
#   fill value 9.969e36 is NOT auto-masked   -> drop
#   negative concentrations occur            -> drop
#   timestamps are UTC, China is UTC+8       -> convert
#   duplicate and out-of-month timestamps    -> drop, keep the file's month
#   coverage varies                          -> require >=8 valid hours/day

import urllib.request, os, re, xarray as xr

MONTHS = ["201312","201401","201402","201412","201501","201502",
          "201512","201601","201602","201612","201701","201702",
          "201712","201801","201802","201812","201901","201902",
          "201912","202001","202002"]
BASE = "https://zenodo.org/records/6404603/files"
os.makedirs("/content/cn_raw", exist_ok=True)

frames = []
for m in MONTHS:
    p = f"/content/cn_raw/aqi{m}.nc"
    if not os.path.exists(p):
        urllib.request.urlretrieve(f"{BASE}/aqi{m}.nc?download=1", p)

    ds = xr.open_dataset(p)
    area = np.array([a.decode("utf-8","ignore").strip() for a in ds.area.values])
    t = pd.to_datetime([s.decode().strip() for s in ds.time_point.values],
                       format="%Y-%m-%dT%H:%M:%SZ", utc=True)

    PM = ds.pm2_5.values.astype("float64")
    PM[(PM > 1e6) | (PM < 0)] = np.nan

    X = pd.DataFrame(PM, index=t, columns=area)
    X = X[~X.index.duplicated(keep="first")]
    X.index = X.index.tz_convert("Asia/Shanghai")
    X = X[(X.index.year == int(m[:4])) & (X.index.month == int(m[4:]))]  # drop strays

    # average stations within a city, then require >=8 valid hours per day
    hourly = X.T.groupby(level=0).mean().T
    d = hourly.groupby(hourly.index.normalize())
    daily, hrs = d.mean(), d.count()
    frames.append(daily.where(hrs >= 8))
    ds.close()
    print(f"  {m}: {daily.shape[1]} cities, {len(daily)} days")

CN = pd.concat(frames).sort_index()
CN.index = CN.index.tz_localize(None)
CN.index.name = "date"

print(f"\ncities: {CN.shape[1]}   days: {len(CN)}")
print(f"span: {CN.index.min().date()} to {CN.index.max().date()}")
print(f"non-missing city-days: {100*CN.notna().mean().mean():.1f}%")
for c in ["Shanghai","Shenzhen","Beijing"]:
    s = CN[c].dropna()
    print(f"  {c:<10}{len(s):>5} days   mean {s.mean():5.1f}   max {s.max():6.1f}")

  201312: 360 cities, 31 days
  201401: 360 cities, 31 days
  201402: 360 cities, 28 days
  201412: 360 cities, 31 days
  201501: 360 cities, 31 days
  201502: 360 cities, 28 days
  201512: 360 cities, 31 days
  201601: 360 cities, 31 days
  201602: 360 cities, 29 days
  201612: 360 cities, 31 days
  201701: 360 cities, 31 days
  201702: 360 cities, 28 days
  201712: 360 cities, 31 days
  201801: 360 cities, 31 days
  201802: 360 cities, 28 days
  201812: 360 cities, 26 days
  201901: 365 cities, 31 days
  201902: 365 cities, 28 days
  201912: 360 cities, 31 days
  202001: 360 cities, 31 days
  202002: 360 cities, 29 days

cities: 370   days: 627
span: 2013-12-01 to 2020-02-29
non-missing city-days: 86.1%
  Shanghai    622 days   mean  61.2   max  462.9
  Shenzhen    622 days   mean  39.0   max  127.7
  Beijing     622 days   mean  75.4   max  494.4


In [ ]:
CN.to_csv(f"{DIR}/cn_pm25_daily.csv")

cov = CN.notna().mean().sort_values(ascending=False)
print(f"cities with >=90% coverage: {(cov >= 0.90).sum()}")
print(f"cities with >=50% coverage: {(cov >= 0.50).sum()}")
print(f"cities with <10% coverage:  {(cov < 0.10).sum()}")
print("\nworst 10:")
print((100*cov.tail(10)).round(1).to_string())

cities with >=90% coverage: 179
cities with >=50% coverage: 358
cities with <10% coverage:  10

worst 10:
Bozhou_xj     9.4
Bozhou_ah     9.4
Suzhou_ah     9.4
Fuzhou_jx     9.4
Fuzhou_fj     9.4
Fuyang_un     9.4
Fuyang_ah     9.4
Suzhou_js     9.4
Taizhou_js    9.4
Taizhou_zj    9.4


In [ ]:
import re
suff = [c for c in CN.columns if re.search(r"_[a-z]{2}$", c)]
print(f"{len(suff)} suffixed names:", suff)

for c in suff:
    base = c.rsplit("_", 1)[0]
    if base in CN.columns:
        a, b = CN[base], CN[c]
        overlap = (a.notna() & b.notna()).sum()
        print(f"\n{base} ({a.notna().sum()}d) vs {c} ({b.notna().sum()}d)"
              f"  overlapping days: {overlap}")
        if overlap:
            print(f"   correlation on overlap: {a.corr(b):.3f}")

10 suffixed names: ['Bozhou_ah', 'Bozhou_xj', 'Fuyang_ah', 'Fuyang_un', 'Fuzhou_fj', 'Fuzhou_jx', 'Suzhou_ah', 'Suzhou_js', 'Taizhou_js', 'Taizhou_zj']

Bozhou (444d) vs Bozhou_ah (59d)  overlapping days: 0

Bozhou (444d) vs Bozhou_xj (59d)  overlapping days: 0

Fuyang (532d) vs Fuyang_ah (59d)  overlapping days: 0

Fuyang (532d) vs Fuyang_un (59d)  overlapping days: 0

Fuzhou (563d) vs Fuzhou_fj (59d)  overlapping days: 0

Fuzhou (563d) vs Fuzhou_jx (59d)  overlapping days: 0

Suzhou (563d) vs Suzhou_ah (59d)  overlapping days: 0

Suzhou (563d) vs Suzhou_js (59d)  overlapping days: 0

Taizhou (563d) vs Taizhou_js (59d)  overlapping days: 0

Taizhou (563d) vs Taizhou_zj (59d)  overlapping days: 0


In [ ]:
AMBIG = ["Bozhou","Fuyang","Fuzhou","Suzhou","Taizhou"]
drop = [c for c in CN.columns if c.rsplit("_",1)[0] in AMBIG]
print(f"dropping {len(drop)}: {drop}")

CN = CN.drop(columns=drop)
CN.to_csv(f"{DIR}/cn_pm25_daily.csv")

cov = CN.notna().mean()
print(f"\ncities remaining: {CN.shape[1]}")
print(f"  >=90% coverage: {(cov >= 0.90).sum()}")
print(f"  >=50% coverage: {(cov >= 0.50).sum()}")
for c in ["Shanghai","Shenzhen","Beijing"]:
    print(f"  {c}: {100*cov[c]:.1f}% — present")

dropping 15: ['Bozhou', 'Fuyang', 'Fuzhou', 'Suzhou', 'Taizhou', 'Bozhou_ah', 'Bozhou_xj', 'Fuyang_ah', 'Fuyang_un', 'Fuzhou_fj', 'Fuzhou_jx', 'Suzhou_ah', 'Suzhou_js', 'Taizhou_js', 'Taizhou_zj']

cities remaining: 355
  >=90% coverage: 179
  >=50% coverage: 353
  Shanghai: 99.2% — present
  Shenzhen: 99.2% — present
  Beijing: 99.2% — present


In [ ]:
# ── inspect the A-H structure on one pair ──────────────────────────
!pip install akshare -q
import akshare as ak

ah = ak.stock_zh_ah_name()
print(f"pairs: {len(ah)}")
print(ah.head(5).to_string())

# CCB: H-share code 00939
h = ak.stock_zh_ah_daily(symbol="00939", start_year="2013", end_year="2020")
print(f"\ndaily columns: {list(h.columns)}")
print(f"rows: {len(h)}   {h.iloc[0,0]} to {h.iloc[-1,0]}")
print(h.head(3).to_string())

# does the function return the A side, the H side, or both?
print("\nclose price range:", h.iloc[:,2].min(), "to", h.iloc[:,2].max())

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 33.0 MB/s eta 0:00:00


pairs: 220
      代码       名称
0  00525   广深铁路股份
1  02238     广汽集团
2  03678     弘业期货
3  02068     中铝国际
4  00323  马鞍山钢铁股份



daily columns: ['日期', '开盘', '收盘', '最高', '最低', '成交量']
rows: 3446   2013-01-02 to 2020-12-31
           日期    开盘    收盘    最高    最低          成交量
0  2013-01-02  6.29  6.43  6.45  6.29  328984520.0
1  2013-01-03  6.50  6.50  6.50  6.41  343210370.0
2  2013-01-04  6.52  6.50  6.52  6.42  258003862.0

close price range: 4.31 to 9.12


In [ ]:
d = h.iloc[:, 0]
print("duplicate dates:", d.duplicated().sum())
print("unique dates:", d.nunique(), "of", len(d))
print("\nfirst 6 rows:")
print(h.head(6).to_string())

# if dates repeat, the two series are stacked
if d.duplicated().sum() > 0:
    g = h.groupby(h.columns[0]).size()
    print("\nrows per date:", g.value_counts().to_dict())
    ex = d[d.duplicated(keep=False)].iloc[0]
    print(f"\nboth rows for {ex}:")
    print(h[d == ex].to_string())

duplicate dates: 1477
unique dates: 1969 of 3446

first 6 rows:
           日期    开盘    收盘    最高    最低          成交量
0  2013-01-02  6.29  6.43  6.45  6.29  328984520.0
1  2013-01-03  6.50  6.50  6.50  6.41  343210370.0
2  2013-01-04  6.52  6.50  6.52  6.42  258003862.0
3  2013-01-07  6.51  6.49  6.52  6.43  208295234.0
4  2013-01-08  6.48  6.38  6.49  6.35  261603328.0
5  2013-01-09  6.36  6.43  6.46  6.36  201998020.0

rows per date: {2: 1477, 1: 492}

both rows for 2014-01-02:
             日期   开盘    收盘   最高    最低          成交量
244  2014-01-02  5.9  5.82  5.9  5.76  198808481.0
491  2014-01-02  5.9  5.82  5.9  5.76  198808481.0


In [ ]:
h = h.drop_duplicates(subset=h.columns[0]).reset_index(drop=True)
print(f"H side after dedup: {len(h)} rows, {h.iloc[0,0]} to {h.iloc[-1,0]}")

# CCB's A-share is 601939 on the Shanghai exchange
try:
    a = ak.stock_zh_a_hist(symbol="601939", period="daily",
                           start_date="20130101", end_date="20201231", adjust="")
    print(f"\nA side: {len(a)} rows")
    print(f"columns: {list(a.columns)}")
    print(a.head(3).to_string())
    print(f"\nJan 2013 close ~ {a.iloc[0,2]}  (A-share should be ~4.5 RMB)")
    print(f"Jan 2013 H close ~ {h.iloc[0,2]}  (H-share should be ~6.4 HKD)")
except Exception as e:
    print("stock_zh_a_hist failed:", type(e).__name__, str(e)[:200])
    print("\navailable A-share functions:")
    print([f for f in dir(ak) if f.startswith("stock_zh_a_") and "hist" in f][:20])

H side after dedup: 1969 rows, 2013-01-02 to 2020-12-31

A side: 1945 rows
columns: ['日期', '股票代码', '开盘', '收盘', '最高', '最低', '成交量', '成交额', '振幅', '涨跌幅', '涨跌额', '换手率']
           日期    股票代码    开盘    收盘    最高    最低     成交量          成交额    振幅   涨跌幅   涨跌额   换手率
0  2013-01-04  601939  4.64  4.70  4.77  4.64  741390  347513152.0  2.83  2.17  0.10  0.77
1  2013-01-07  601939  4.69  4.73  4.74  4.68  405054  191167880.0  1.28  0.64  0.03  0.42
2  2013-01-08  601939  4.72  4.65  4.74  4.62  520824  242711654.0  2.54 -1.69 -0.08  0.54

Jan 2013 close ~ 4.64  (A-share should be ~4.5 RMB)
Jan 2013 H close ~ 6.43  (H-share should be ~6.4 HKD)


In [ ]:
cands = [f for f in dir(ak) if "_ah_" in f]
print("A-H functions:", cands)

for fn in cands:
    if fn in ("stock_zh_ah_daily", "stock_zh_ah_name"):
        continue
    try:
        r = getattr(ak, fn)()
        print(f"\n=== {fn} ===  {r.shape}")
        print(list(r.columns))
        print(r.head(3).to_string())
    except Exception as e:
        print(f"\n{fn}: {type(e).__name__} {str(e)[:120]}")

A-H functions: ['stock_zh_ah_daily', 'stock_zh_ah_name', 'stock_zh_ah_spot', 'stock_zh_ah_spot_em']



=== stock_zh_ah_spot ===  (220, 13)
['代码', '名称', '最新价', '涨跌幅', '涨跌额', '买入', '卖出', '成交量', '成交额', '今开', '昨收', '最高', '最低']
      代码      名称   最新价   涨跌幅   涨跌额    买入    卖出         成交量         成交额    今开    昨收    最高    最低
0  00525  广深铁路股份  2.25  1.35  0.03  2.25  2.25   5134000.0  11593089.0  2.19  2.22  2.29  2.19
1  02238    广汽集团  2.20  3.77  0.08  2.20  2.20  22253308.0  48906918.2  2.15  2.12  2.24  2.13
2  03678    弘业期货  2.19  0.46  0.01  2.19  2.19   3294000.0   7254060.0  2.21  2.17  2.23  2.18



=== stock_zh_ah_spot_em ===  (202, 10)
['序号', '名称', 'H股代码', '最新价-HKD', 'H股-涨跌幅', 'A股代码', '最新价-RMB', 'A股-涨跌幅', '比价', '溢价']
   序号    名称   H股代码  最新价-HKD  H股-涨跌幅    A股代码  最新价-RMB  A股-涨跌幅    比价      溢价
0   2   康希诺  06185    32.06   37.36  688185    64.43   20.00  2.35  134.50
1   3  康龙化成  03759    33.86    7.70  300759    50.90    5.06  1.75   75.41
2   4  复星医药  02196    17.90    6.93  600196    24.84    5.12  1.62   61.93


/usr/local/lib/python3.12/dist-packages/akshare/stock/stock_hsgt_em.py:67: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  temp_df["最新价-HKD"] = pd.to_numeric(temp_df["最新价-HKD"], errors="coerce") / 1000
/usr/local/lib/python3.12/dist-packages/akshare/stock/stock_hsgt_em.py:68: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  temp_df["H股-涨跌幅"] = pd.to_numeric(temp_df["H股-涨跌幅"], errors="coerce") / 100
/usr/local/lib/python3.12/dist-packages/akshare/stock/stock_hsgt_em.py:69: SettingWithCopyWarning: 
A value is tryi

In [ ]:
m = ak.stock_zh_ah_spot_em()[["名称", "H股代码", "A股代码"]].copy()
m.columns = ["name", "h_code", "a_code"]
m["h_code"] = m.h_code.astype(str).str.zfill(5)
m["a_code"] = m.a_code.astype(str).str.zfill(6)

print(f"pairs with both codes: {m.notna().all(axis=1).sum()} of {len(m)}")
print(f"duplicate H codes: {m.h_code.duplicated().sum()}   "
      f"duplicate A codes: {m.a_code.duplicated().sum()}")

# does it contain the pair we verified by hand?
print("\nCCB row:")
print(m[m.h_code == "00939"].to_string())

# A-code prefixes tell us which mainland exchange
print("\nA-code prefixes:")
print(m.a_code.str[:3].value_counts().head(10).to_string())

m.to_csv(f"{DIR}/ah_pairs.csv", index=False)

HTTPError: 502 Server Error: Bad Gateway for url: https://push2.eastmoney.com/api/qt/clist/get?np=1&fltt=1&invt=2&fs=b%3ADLMK0101&fields=f193%2Cf191%2Cf192%2Cf12%2Cf13%2Cf14%2Cf1%2Cf2%2Cf4%2Cf3%2Cf152%2Cf186%2Cf190%2Cf187%2Cf189%2Cf188&fid=f3&pn=1&pz=100&po=1&dect=1&wbp2u=%7C0%7C0%7C0%7Cweb

In [ ]:
import time
for attempt in range(5):
    try:
        raw = ak.stock_zh_ah_spot_em()
        print(f"got it on attempt {attempt+1}: {raw.shape}")
        break
    except Exception as e:
        print(f"attempt {attempt+1}: {type(e).__name__}")
        time.sleep(4)
else:
    raw = None
    print("all attempts failed — try again in a minute")

if raw is not None:
    m = raw[["名称", "H股代码", "A股代码"]].copy()
    m.columns = ["name", "h_code", "a_code"]
    m["h_code"] = m.h_code.astype(str).str.zfill(5)
    m["a_code"] = m.a_code.astype(str).str.zfill(6)

    print(f"\npairs: {len(m)}")
    print(f"duplicate H: {m.h_code.duplicated().sum()}   duplicate A: {m.a_code.duplicated().sum()}")
    print("\nCCB:")
    print(m[m.h_code == "00939"].to_string())
    print("\nA-code prefixes (600/601/603/688 = Shanghai, 000/002/300 = Shenzhen):")
    print(m.a_code.str[:3].value_counts().to_string())

    m.to_csv(f"{DIR}/ah_pairs.csv", index=False)
    print(f"\nsaved to {DIR}/ah_pairs.csv")

attempt 1: HTTPError
attempt 2: HTTPError
attempt 3: HTTPError
attempt 4: HTTPError
attempt 5: HTTPError
all attempts failed — try again in a minute


In [ ]:
# H codes + names from the pair list; A codes + names from the full listing
ah = ak.stock_zh_ah_name()
ah.columns = ["h_code", "name"]
ah["h_code"] = ah.h_code.astype(str).str.zfill(5)
print(f"H-side pairs: {len(ah)}")

try:
    alist = ak.stock_info_a_code_name()
    alist.columns = ["a_code", "name"]
    alist["a_code"] = alist.a_code.astype(str).str.zfill(6)
    print(f"all A-shares: {len(alist)}")
except Exception as e:
    print("A listing failed:", type(e).__name__, str(e)[:150])
    alist = None

if alist is not None:
    m = ah.merge(alist, on="name", how="left")
    ok = m.a_code.notna()
    print(f"\nmatched on exact name: {ok.sum()} of {len(m)}")
    print(f"names matching >1 A-code: {m[ok].h_code.duplicated().sum()}")

    print("\nCCB check (expect 601939):")
    print(m[m.h_code == "00939"].to_string())

    print("\nunmatched, first 15:")
    print(m[~ok].head(15).to_string())

H-side pairs: 220
A listing failed: ConnectionError ('Connection aborted.', ConnectionResetError(104, 'Connection reset by peer'))


In [ ]:
ah.to_csv(f"{DIR}/ah_h_codes.csv", index=False)
print("saved H-side pair list:", len(ah), "rows")

saved H-side pair list: 220 rows


In [ ]:
ah.to_csv(f"{DIR}/ah_h_codes.csv", index=False)
print(f"saved H-side list: {len(ah)} pairs\n")

# does the daily endpoint still work?
try:
    t = ak.stock_zh_ah_daily(symbol="00939", start_year="2013", end_year="2020")
    print(f"daily endpoint alive: {len(t)} rows for CCB")
    alive = True
except Exception as e:
    print("daily endpoint also blocked:", type(e).__name__)
    alive = False

saved H-side list: 220 pairs



daily endpoint alive: 3446 rows for CCB


In [ ]:
# ── H-side daily prices, all 220 pairs ─────────────────────────────
import time, os

OUT = f"{DIR}/h_prices.csv"
done = {}
if os.path.exists(OUT):
    prev = pd.read_csv(OUT, parse_dates=["date"])
    done = {c: g for c, g in prev.groupby("h_code")}
    print(f"resuming: {len(done)} already saved")

rows, failed = [], []
for i, code in enumerate(ah.h_code, 1):
    if code in done:
        rows.append(done[code]); continue
    try:
        d = ak.stock_zh_ah_daily(symbol=code, start_year="2013", end_year="2020")
        d = d.iloc[:, :6]
        d.columns = ["date", "open", "close", "high", "low", "volume"]
        d["date"] = pd.to_datetime(d.date)
        d = d.drop_duplicates("date")
        d["h_code"] = code
        rows.append(d)
    except Exception as e:
        failed.append((code, type(e).__name__))
    if i % 20 == 0:
        pd.concat(rows, ignore_index=True).to_csv(OUT, index=False)
        print(f"  {i}/220   kept {len(rows)}   failed {len(failed)}")
    time.sleep(0.6)

H = pd.concat(rows, ignore_index=True)
H.to_csv(OUT, index=False)

print(f"\nseries: {H.h_code.nunique()} of 220")
print(f"rows: {len(H):,}   {H.date.min().date()} to {H.date.max().date()}")
n = H.groupby("h_code").size()
print(f"days per series: median {n.median():.0f}, min {n.min()}, max {n.max()}")
if failed:
    print(f"\nfailed ({len(failed)}): {failed[:10]}")

  20/220   kept 20   failed 0


  40/220   kept 24   failed 16


  60/220   kept 33   failed 27


  80/220   kept 42   failed 38


  100/220   kept 55   failed 45


  120/220   kept 69   failed 51


  140/220   kept 83   failed 57


  160/220   kept 99   failed 61


  180/220   kept 119   failed 61


  200/220   kept 138   failed 62


  220/220   kept 158   failed 62

series: 140 of 220
rows: 265,942   2013-01-02 to 2020-12-31
days per series: median 1969, min 6, max 3938

failed (62): [('03308', 'KeyError'), ('03750', 'KeyError'), ('03986', 'KeyError'), ('09630', 'KeyError'), ('01377', 'KeyError'), ('06809', 'KeyError'), ('02476', 'KeyError'), ('02676', 'KeyError'), ('06821', 'KeyError'), ('01989', 'KeyError')]


In [ ]:
n = H.groupby("h_code").size().sort_values(ascending=False)
print("series with >2500 rows:", (n > 2500).sum())
print(n.head(8).to_string())

dup = H.duplicated(["h_code","date"]).sum()
print(f"\nduplicate (code,date) pairs: {dup}")

H = H.drop_duplicates(["h_code","date"])
n2 = H.groupby("h_code").size()
print(f"\nafter dedup: {H.h_code.nunique()} series, {len(H):,} rows")
print(f"days per series: median {n2.median():.0f}, min {n2.min()}, max {n2.max()}")
print(f"series with <500 days: {(n2 < 500).sum()}")

H.to_csv(OUT, index=False)

series with >2500 rows: 15
h_code
00323    3938
00347    3938
00338    3938
02880    3938
02238    3938
02068    3938
01618    3938
01033    3938

duplicate (code,date) pairs: 32281

after dedup: 140 series, 233,661 rows
days per series: median 1969, min 6, max 1969
series with <500 days: 9


In [ ]:
print("CCB present:", "00939" in set(H.h_code))

n2 = H.groupby("h_code").size()
short = n2[n2 < 500]
print(f"\nseries under 500 days ({len(short)}):")
print(short.to_string())

# how much of the sample falls in the winter months the pollution data covers?
w = H[H.date.dt.month.isin([12,1,2])]
print(f"\nwinter rows: {len(w):,} of {len(H):,} ({100*len(w)/len(H):.0f}%)")
print(f"winter days per series: median {w.groupby('h_code').size().median():.0f}")

CCB present: True

series under 500 days (9):
h_code
01877    498
03347    100
03759    270
06185    436
06690      6
06806    418
09969    193
09989    122
09995     38

winter rows: 56,440 of 233,661 (24%)
winter days per series: median 477


In [ ]:
H.to_csv(OUT, index=False)
print("saved:", OUT, len(H), "rows")

saved: /content/drive/MyDrive/pollution-ah/h_prices.csv 233661 rows


In [ ]:
import time
raw = None
for attempt in range(3):
    try:
        raw = ak.stock_zh_ah_spot_em()
        print(f"success on attempt {attempt+1}: {raw.shape}")
        break
    except Exception as e:
        print(f"attempt {attempt+1}: {type(e).__name__}")
        time.sleep(20)

if raw is not None:
    m = raw[["名称", "H股代码", "A股代码"]].copy()
    m.columns = ["name", "h_code", "a_code"]
    m["h_code"] = m.h_code.astype(str).str.zfill(5)
    m["a_code"] = m.a_code.astype(str).str.zfill(6)
    m.to_csv(f"{DIR}/ah_pairs.csv", index=False)

    print(f"\npairs: {len(m)}")
    print("CCB (expect 601939):")
    print(m[m.h_code == "00939"].to_string())
    print("\nA-code prefixes (6xx = Shanghai, 0xx/3xx = Shenzhen):")
    print(m.a_code.str[:1].value_counts().to_string())
    have = set(H.h_code)
    print(f"\nmapped pairs where we already have H prices: {m.h_code.isin(have).sum()}")

attempt 1: HTTPError
attempt 2: ConnectionError
attempt 3: HTTPError


In [ ]:
!pip install cdsapi -q

CDS_KEY = ""     # paste your CDS key here, run once, then clear it again

with open("/root/.cdsapirc", "w") as f:
    f.write("url: https://cds.climate.copernicus.eu/api\n")
    f.write(f"key: {CDS_KEY}\n")
print("credentials written")

credentials written


In [ ]:
import cdsapi, os
os.makedirs(f"{DIR}/era5", exist_ok=True)
c = cdsapi.Client()

VARS = ["boundary_layer_height",
        "10m_u_component_of_wind", "10m_v_component_of_wind",
        "surface_pressure", "2m_temperature", "total_precipitation",
        "total_cloud_cover", "surface_solar_radiation_downwards"]

BOX = {"shanghai": [31.50, 121.25, 31.00, 121.75],
       "hongkong": [22.50, 114.00, 22.00, 114.50]}

def pull(city, year, months):
    out = f"{DIR}/era5/era5_{city}_{year}.nc"
    if os.path.exists(out):
        print(f"  exists: {out.split('/')[-1]}"); return
    c.retrieve("reanalysis-era5-single-levels", {
        "product_type": "reanalysis",
        "variable": VARS,
        "year": str(year),
        "month": months,
        "day": [f"{d:02d}" for d in range(1, 32)],
        "time": [f"{h:02d}:00" for h in range(24)],
        "area": BOX[city],
        "data_format": "netcdf",
        "download_format": "unarchived",
    }, out)
    print(f"  saved {out.split('/')[-1]}  {os.path.getsize(out)/1e6:.1f} MB")

# one test: Shanghai, December 2018 only
pull("shanghai", 2018, ["12"])

In [1]:
import cdsapi
c = cdsapi.Client(retry_max=1)     # fail fast instead of retrying

try:
    c.retrieve("reanalysis-era5-single-levels", {
        "product_type": "reanalysis",
        "variable": ["2m_temperature"],
        "year": "2018", "month": "12", "day": "01",
        "time": "12:00",
        "area": [31.50, 121.25, 31.00, 121.75],
        "data_format": "netcdf",
    }, "/content/test.nc")
    import os
    print("OK —", os.path.getsize("/content/test.nc"), "bytes")
except Exception as e:
    print(type(e).__name__, str(e)[:400])

ModuleNotFoundError: No module named 'cdsapi'

In [2]:
!pip install cdsapi -q

CDS_KEY = ""     # paste your key, run, then clear it again

with open("/root/.cdsapirc", "w") as f:
    f.write("url: https://cds.climate.copernicus.eu/api\n")
    f.write(f"key: {CDS_KEY}\n")
print("credentials written")

credentials written


In [3]:
import cdsapi, os
c = cdsapi.Client(retry_max=1)
try:
    c.retrieve("reanalysis-era5-single-levels", {
        "product_type": "reanalysis",
        "variable": ["2m_temperature"],
        "year": "2018", "month": "12", "day": "01",
        "time": "12:00",
        "area": [31.50, 121.25, 31.00, 121.75],
        "data_format": "netcdf",
    }, "/content/test.nc")
    print("OK —", os.path.getsize("/content/test.nc"), "bytes")
except Exception as e:
    print(type(e).__name__, str(e)[:400])

HTTPError 500 Server Error: Internal Server Error for url: https://cds.climate.copernicus.eu/api/retrieve/v1/processes/reanalysis-era5-single-levels/execution


In [4]:
import time
print("=== A-code mapping (Eastmoney) ===")
try:
    raw = ak.stock_zh_ah_spot_em()
    print("ALIVE:", raw.shape)
except Exception as e:
    raw = None
    print("blocked:", type(e).__name__)

print("\n=== CDS ===")
try:
    import cdsapi, os
    c = cdsapi.Client(retry_max=1)
    c.retrieve("reanalysis-era5-single-levels", {
        "product_type": "reanalysis", "variable": ["2m_temperature"],
        "year": "2018", "month": "12", "day": "01", "time": "12:00",
        "area": [31.50, 121.25, 31.00, 121.75], "data_format": "netcdf",
    }, "/content/test.nc")
    print("ALIVE:", os.path.getsize("/content/test.nc"), "bytes")
except Exception as e:
    print("blocked:", type(e).__name__, str(e)[:120])

=== A-code mapping (Eastmoney) ===
blocked: NameError

=== CDS ===
blocked: HTTPError 500 Server Error: Internal Server Error for url: https://cds.climate.copernicus.eu/api/retrieve/v1/processes/reanalysis-


In [5]:
!pip install akshare cdsapi -q
from google.colab import drive
drive.mount('/content/drive')

import glob, numpy as np, pandas as pd, akshare as ak
DIR = "/content/drive/MyDrive/pollution-ah"

CN = pd.read_csv(f"{DIR}/cn_pm25_daily.csv", index_col="date", parse_dates=True)
H  = pd.read_csv(f"{DIR}/h_prices.csv", parse_dates=["date"],
                 dtype={"h_code": str})
ah = pd.read_csv(f"{DIR}/ah_h_codes.csv", dtype={"h_code": str})
print(f"CN {CN.shape}   H {H.h_code.nunique()} series   ah {len(ah)} codes")

try:
    raw = ak.stock_zh_ah_spot_em()
    print("\nA-code mapping ALIVE:", raw.shape)
except Exception as e:
    raw = None
    print("\nA-code mapping blocked:", type(e).__name__)

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 67.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 71.8 MB/s eta 0:00:00
Mounted at /content/drive
CN (627, 355)   H 140 series   ah 220 codes



A-code mapping blocked: HTTPError


In [6]:
funcs = [f for f in dir(ak) if f.startswith("stock_info_")]
print("candidates:", funcs)

alist = None
for fn in ["stock_info_sh_name_code", "stock_info_sz_name_code",
           "stock_info_a_code_name"]:
    if not hasattr(ak, fn):
        continue
    try:
        r = getattr(ak, fn)()
        print(f"\n{fn}: OK  {r.shape}")
        print("  columns:", list(r.columns))
        print(r.head(3).to_string())
    except Exception as e:
        print(f"\n{fn}: {type(e).__name__} {str(e)[:100]}")

candidates: ['stock_info_a_code_name', 'stock_info_bj_name_code', 'stock_info_change_name', 'stock_info_cjzc_em', 'stock_info_global_cls', 'stock_info_global_em', 'stock_info_global_futu', 'stock_info_global_sina', 'stock_info_global_ths', 'stock_info_sh_delist', 'stock_info_sh_name_code', 'stock_info_sz_change_name', 'stock_info_sz_delist', 'stock_info_sz_name_code']

stock_info_sh_name_code: OK  (1699, 6)
  columns: ['证券代码', '证券简称', '证券全称', '公司简称', '公司全称', '上市日期']
     证券代码  证券简称  证券全称  公司简称            公司全称        上市日期
0  600000  浦发银行  浦发银行  浦发银行  上海浦东发展银行股份有限公司  1999-11-10
1  600004  白云机场  白云机场  白云机场  广州白云国际机场股份有限公司  2003-04-28
2  600006  东风股份  东风股份  东风股份      东风汽车股份有限公司  1999-07-27

stock_info_sz_name_code: ConnectionError ('Connection aborted.', ConnectionResetError(104, 'Connection reset by peer'))



stock_info_a_code_name: OK  (5549, 2)
  columns: ['code', 'name']
     code   name
0  000001   平安银行
1  000002  万  科Ａ
2  000006   深振业Ａ


In [7]:
alist.to_csv(f"{DIR}/a_listing.csv", index=False)
print("saved A listing")

A = alist.copy()
A.columns = ["a_code", "name"]
A["a_code"] = A.a_code.astype(str).str.zfill(6)
A["key"] = A.name.str.replace(r"\s+", "", regex=True).str.replace("Ａ", "A")

ah["key"] = ah.name.str.replace(r"\s+", "", regex=True)

m = ah.merge(A[["a_code", "key"]], on="key", how="left")
ok = m.a_code.notna()
print(f"\nexact-name matches: {ok.sum()} of {len(m)}")
print(f"H codes matching >1 A code: {m[ok].h_code.duplicated().sum()}")

print("\nCCB (expect 601939):")
print(m[m.h_code == "00939"][["h_code","name","a_code"]].to_string())

have = set(H.h_code)
usable = m[ok & m.h_code.isin(have)]
print(f"\nmatched AND we have H prices: {len(usable)}")
print(f"  Shanghai A-codes (6xx): {usable.a_code.str.startswith('6').sum()}")
print(f"  Shenzhen A-codes (0/3): {(~usable.a_code.str.startswith('6')).sum()}")

print("\nunmatched, first 12:")
print(m[~ok][["h_code","name"]].head(12).to_string())

AttributeError: 'NoneType' object has no attribute 'to_csv'

In [8]:
alist = ak.stock_info_a_code_name()
alist.to_csv(f"{DIR}/a_listing.csv", index=False)
print(f"saved {len(alist)} A-shares")

saved 5549 A-shares


In [9]:
A = alist.copy()
A.columns = ["a_code", "name"]
A["a_code"] = A.a_code.astype(str).str.zfill(6)
A["key"] = A.name.str.replace(r"\s+", "", regex=True).str.replace("Ａ", "A")

ah["key"] = ah.name.str.replace(r"\s+", "", regex=True)

m = ah.merge(A[["a_code", "key"]], on="key", how="left")
ok = m.a_code.notna()
print(f"exact-name matches: {ok.sum()} of {len(m)}")
print(f"H codes matching >1 A code: {m[ok].h_code.duplicated().sum()}")

print("\nCCB (expect 601939):")
print(m[m.h_code == "00939"][["h_code","name","a_code"]].to_string())

have = set(H.h_code)
usable = m[ok & m.h_code.isin(have)]
print(f"\nmatched AND have H prices: {len(usable)}")
print(f"  Shanghai (6xx): {usable.a_code.str.startswith('6').sum()}")
print(f"  Shenzhen (0/3): {(~usable.a_code.str.startswith('6')).sum()}")

print("\nunmatched, first 12:")
print(m[~ok][["h_code","name"]].head(12).to_string())

exact-name matches: 157 of 220
H codes matching >1 A code: 9

CCB (expect 601939):
    h_code  name  a_code
135  00939  建设银行  601939

matched AND have H prices: 99
  Shanghai (6xx): 75
  Shenzhen (0/3): 24

unmatched, first 12:
   h_code      name
0   00525    广深铁路股份
4   00323   马鞍山钢铁股份
6   01375      中州证券
9   00338  上海石油化工股份
10  03996    中国能源建设
11  01528     红星美凯龙
14  01053    重庆钢铁股份
15  00588  北京北辰实业股份
17  01033     中石化油服
18  01812      晨鸣纸业
30  06869    长飞光纤光缆
42  01211     比亚迪股份


In [10]:
def norm(s):
    s = s.str.replace(r"\s+", "", regex=True).str.replace("Ａ", "A")
    return s.str.replace(r"股份$", "", regex=True)

A["key2"] = norm(A.name)
ah["key2"] = norm(ah.name)

m2 = ah.merge(A[["a_code","key2"]], on="key2", how="left")
ok2 = m2.a_code.notna()
print(f"matches after stripping 股份: {ok2.sum()} of {len(m2)}  (was 157)")

dups = m2[ok2].h_code.duplicated(keep=False)
print(f"\nH codes with >1 A match ({m2[ok2][dups].h_code.nunique()} codes):")
print(m2[ok2][dups][["h_code","name","a_code"]].to_string())

have = set(H.h_code)
usable2 = m2[ok2 & m2.h_code.isin(have) & ~m2.h_code.duplicated(keep=False)]
print(f"\nunambiguous AND have H prices: {len(usable2)}")
print(f"  Shanghai: {usable2.a_code.str.startswith('6').sum()}")
print(f"  Shenzhen: {(~usable2.a_code.str.startswith('6')).sum()}")

print("\nstill unmatched:")
print(m2[~ok2]

SyntaxError: incomplete input (3072555754.py, line 23)

In [11]:
def norm(s):
    s = s.str.replace(r"\s+", "", regex=True).str.replace("Ａ", "A")
    return s.str.replace(r"股份$", "", regex=True)

A["key2"] = norm(A.name)
ah["key2"] = norm(ah.name)

m2 = ah.merge(A[["a_code", "key2"]], on="key2", how="left")
ok2 = m2.a_code.notna()
print(f"matches after stripping 股份: {ok2.sum()} of {len(m2)}  (was 157)")

dups = m2[ok2].h_code.duplicated(keep=False)
print(f"\nH codes with more than one A match: {m2[ok2][dups].h_code.nunique()}")
print(m2[ok2][dups][["h_code", "name", "a_code"]].to_string())

have = set(H.h_code)
usable2 = m2[ok2 & m2.h_code.isin(have) & ~m2.h_code.duplicated(keep=False)]
print(f"\nunambiguous and have H prices: {len(usable2)}")
print(f"  Shanghai: {usable2.a_code.str.startswith('6').sum()}")
print(f"  Shenzhen: {(~usable2.a_code.str.startswith('6')).sum()}")

print("\nstill unmatched:")
print(m2.loc[~ok2, ["h_code", "name"]].to_string())

matches after stripping 股份: 166 of 220  (was 157)

H codes with more than one A match: 11
    h_code    name  a_code
0    00525  广深铁路股份  601333
1    02238    广汽集团  601238
2    03678    弘业期货  001236
3    02068    中铝国际  601068
5    00895    东江环保  002672
7    01618    中国中冶  601618
8    00347    鞍钢股份  000898
12   02866    中远海发  601866
13   06196    郑州银行  002936
14   01053  重庆钢铁股份  601005
16   02880    辽港股份  601880
202  00525  广深铁路股份  601333
203  02238    广汽集团  601238
204  03678    弘业期货  001236
205  02068    中铝国际  601068
207  00895    东江环保  002672
209  01618    中国中冶  601618
210  00347    鞍钢股份  000898
214  02866    中远海发  601866
215  06196    郑州银行  002936
216  01053  重庆钢铁股份  601005
218  02880    辽港股份  601880

unambiguous and have H prices: 85
  Shanghai: 68
  Shenzhen: 17

still unmatched:
    h_code      name
4    00323   马鞍山钢铁股份
6    01375      中州证券
9    00338  上海石油化工股份
10   03996    中国能源建设
11   01528     红星美凯龙
15   00588  北京北辰实业股份
17   01033     中石化油服
18   01812      晨鸣纸业
30   06869    长飞光

In [12]:
print("ah rows:", len(ah), " unique h_code:", ah.h_code.nunique())
print("duplicated h_code rows:", ah.h_code.duplicated().sum())

s = ah.loc[ah.name.str.contains("马鞍山"), "name"]
print("\nraw repr:", [repr(x) for x in s.unique()])
print("after norm:", [repr(x) for x in norm(s).unique()])

hit = A[A.name.str.contains("马鞍山")]
print("\nA-listing candidates:")
print(hit[["a_code","name","key2"]].to_string())

ah rows: 220  unique h_code: 202
duplicated h_code rows: 18

raw repr: ["'马鞍山钢铁股份'"]
after norm: ["'马鞍山钢铁'"]

A-listing candidates:
Empty DataFrame
Columns: [a_code, name, key2]
Index: []


In [13]:
ah = ah.drop_duplicates("h_code").reset_index(drop=True)
ah["key2"] = norm(ah.name)
print(f"ah now: {len(ah)} unique pairs")

m2 = ah.merge(A[["a_code","key2"]], on="key2", how="left")
ok2 = m2.a_code.notna()
print(f"name matches: {ok2.sum()} of {len(m2)}")

dups = m2[ok2].h_code.duplicated(keep=False)
print(f"ambiguous (one H, several A): {m2[ok2][dups].h_code.nunique()}")

have = set(H.h_code)
final = m2[ok2 & ~m2.h_code.duplicated(keep=False) & m2.h_code.isin(have)]
print(f"\nFINAL usable pairs: {len(final)}")
print(f"  Shanghai: {final.a_code.str.startswith('6').sum()}")
print(f"  Shenzhen: {(~final.a_code.str.startswith('6')).sum()}")

final[["h_code","name","a_code"]].to_csv(f"{DIR}/ah_pairs.csv", index=False)
print(f"\nsaved ah_pairs.csv")

ah now: 202 unique pairs
name matches: 155 of 202
ambiguous (one H, several A): 0

FINAL usable pairs: 96
  Shanghai: 75
  Shenzhen: 21

saved ah_pairs.csv


In [14]:
OUTA = f"{DIR}/a_prices.csv"
rows, failed = [], []

for i, (hc, ac) in enumerate(zip(final.h_code, final.a_code), 1):
    try:
        d = ak.stock_zh_a_hist(symbol=ac, period="daily",
                               start_date="20130101", end_date="20201231",
                               adjust="hfq")          # back-adjusted for splits
        d = d[["日期","开盘","收盘","最高","最低","成交量"]]
        d.columns = ["date","open","close","high","low","volume"]
        d["date"] = pd.to_datetime(d.date)
        d["a_code"], d["h_code"] = ac, hc
        rows.append(d.drop_duplicates("date"))
    except Exception as e:
        failed.append((ac, type(e).__name__))
    if i % 20 == 0:
        pd.concat(rows).to_csv(OUTA, index=False)
        print(f"  {i}/{len(final)}  kept {len(rows)}  failed {len(failed)}")
    time.sleep(0.6)

Ap = pd.concat(rows, ignore_index=True).drop_duplicates(["a_code","date"])
Ap.to_csv(OUTA, index=False)
print(f"\nA series: {Ap.a_code.nunique()} of {len(final)}")
print(f"rows: {len(Ap):,}   {Ap.date.min().date()} to {Ap.date.max().date()}")
if failed: print(f"failed: {failed[:8]}")

  20/96  kept 2  failed 18
  40/96  kept 2  failed 38
  60/96  kept 2  failed 58
  80/96  kept 2  failed 78

A series: 2 of 96
rows: 3,515   2013-01-04 to 2020-12-31
failed: [('601238', 'ConnectionError'), ('001236', 'ConnectionError'), ('601068', 'ConnectionError'), ('002672', 'ConnectionError'), ('601618', 'ConnectionError'), ('000898', 'ConnectionError'), ('601866', 'ConnectionError'), ('002936', 'ConnectionError')]


In [15]:
import os
OUTA = f"{DIR}/a_prices.csv"
done = set()
if os.path.exists(OUTA):
    done = set(pd.read_csv(OUTA, dtype={"a_code": str}).a_code.unique())
    print(f"resuming: {len(done)} already saved")

rows = [pd.read_csv(OUTA, parse_dates=["date"], dtype={"a_code": str, "h_code": str})] if done else []
failed = []

for i, (hc, ac) in enumerate(zip(final.h_code, final.a_code), 1):
    if ac in done:
        continue
    try:
        d = ak.stock_zh_a_hist(symbol=ac, period="daily",
                               start_date="20130101", end_date="20201231",
                               adjust="hfq")
        d = d[["日期","开盘","收盘","最高","最低","成交量"]]
        d.columns = ["date","open","close","high","low","volume"]
        d["date"] = pd.to_datetime(d.date)
        d["a_code"], d["h_code"] = ac, hc
        rows.append(d.drop_duplicates("date"))
        pd.concat(rows, ignore_index=True).to_csv(OUTA, index=False)
    except Exception as e:
        failed.append(ac)
    if i % 10 == 0:
        got = pd.concat(rows).a_code.nunique() if rows else 0
        print(f"  {i}/{len(final)}  have {got}  failed {len(failed)}")
    time.sleep(5)

resuming: 2 already saved
  20/96  have 2  failed 18
  30/96  have 2  failed 28
  40/96  have 2  failed 38
  50/96  have 2  failed 48
  60/96  have 2  failed 58
  70/96  have 2  failed 68
  80/96  have 2  failed 78
  90/96  have 2  failed 88


In [16]:
try:
    d = ak.stock_zh_a_daily(symbol="sh601939", start_date="20130101",
                            end_date="20201231", adjust="hfq")
    print("Sina route ALIVE:", len(d), "rows")
    print(d.head(3).to_string())
except Exception as e:
    print("Sina route:", type(e).__name__, str(e)[:150])

Sina route ALIVE: 1945 rows
         date  open  high   low  close      volume       amount  outstanding_share  turnover
0  2013-01-04  5.70  5.86  5.70   5.78  74139048.0  347513152.0       9.593658e+09  0.007728
1  2013-01-07  5.77  5.83  5.75   5.81  40505416.0  191167872.0       9.593658e+09  0.004222
2  2013-01-08  5.80  5.83  5.68   5.72  52082440.0  242711648.0       9.593658e+09  0.005429


In [17]:
import os, time
OUTA = f"{DIR}/a_prices.csv"

rows, failed, done = [], [], set()
for i, (hc, ac) in enumerate(zip(final.h_code, final.a_code), 1):
    sym = ("sh" if ac.startswith("6") else "sz") + ac
    try:
        d = ak.stock_zh_a_daily(symbol=sym, start_date="20130101",
                                end_date="20201231", adjust="hfq")
        d = d[["date","open","high","low","close","volume"]].copy()
        d["date"] = pd.to_datetime(d.date)
        d["a_code"], d["h_code"] = ac, hc
        rows.append(d.drop_duplicates("date"))
        done.add(ac)
    except Exception as e:
        failed.append((sym, type(e).__name__))
    if i % 15 == 0:
        if rows: pd.concat(rows, ignore_index=True).to_csv(OUTA, index=False)
        print(f"  {i}/{len(final)}  have {len(done)}  failed {len(failed)}")
    time.sleep(1.0)

Ap = pd.concat(rows, ignore_index=True).drop_duplicates(["a_code","date"])
Ap.to_csv(OUTA, index=False)
print(f"\nA series: {Ap.a_code.nunique()} of {len(final)}")
print(f"rows: {len(Ap):,}   {Ap.date.min().date()} to {Ap.date.max().date()}")
n = Ap.groupby("a_code").size()
print(f"days per series: median {n.median():.0f}, min {n.min()}, max {n.max()}")
if failed: print(f"failed ({len(failed)}): {failed[:8]}")

  15/96  have 15  failed 0
  30/96  have 30  failed 0
  45/96  have 45  failed 0
  60/96  have 60  failed 0
  75/96  have 75  failed 0
  90/96  have 90  failed 0

A series: 85 of 96
rows: 133,136   2013-01-04 to 2020-12-31
days per series: median 1918, min 44, max 1945


In [18]:
h = H[H.h_code == "00939"].set_index("date").close
a = Ap[Ap.a_code == "601939"].set_index("date").close

print("CCB 2013-01-04:")
print(f"  H {h.get(pd.Timestamp('2013-01-04'), float('nan')):.2f} HKD")
print(f"  A {a.get(pd.Timestamp('2013-01-04'), float('nan')):.2f} (hfq)")
print("\nCCB 2020-12-31:")
print(f"  H {h.get(pd.Timestamp('2020-12-31'), float('nan')):.2f}")
print(f"  A {a.get(pd.Timestamp('2020-12-31'), float('nan')):.2f}")

# an unadjusted series shows a discrete drop on ex-dividend days;
# an adjusted one does not
r = pd.DataFrame({"h": h, "a": a}).dropna()
r["ratio"] = r.h / r.a
big = r.ratio.pct_change().abs().nlargest(8)
print("\n8 largest one-day moves in the H/A ratio:")
print(pd.DataFrame({"jump_pct": (100*big).round(1),
                    "ratio": r.ratio.reindex(big.index).round(3)}).to_string())

CCB 2013-01-04:
  H 6.50 HKD
  A 5.78 (hfq)

CCB 2020-12-31:
  H 5.89
  A 11.64

8 largest one-day moves in the H/A ratio:
            jump_pct  ratio
date                       
2015-07-06      10.0  0.686
2015-09-01       9.8  0.646
2015-07-07       9.8  0.619
2013-07-02       8.1  0.931
2018-07-09       7.9  0.627
2013-06-13       7.8  0.963
2015-01-19       6.9  0.729
2014-12-16       6.7  0.753


In [19]:
d = ak.stock_zh_ah_daily(symbol="00939", start_year="2013", end_year="2020",
                         adjust="hfq")
print(d.head(3).to_string())
print("\n2013-01-04 close:", d[d.iloc[:,0].astype(str) == "2013-01-04"].iloc[:,2].values)

           日期     开盘     收盘     最高     最低           成交量
0  2012-05-29  6.337  6.423  6.423  6.326  2.556993e+08
1  2012-05-30  6.412  6.359  6.423  6.305  3.751875e+08
2  2012-05-31  6.305  6.498  6.551  6.284  1.112519e+09

2013-01-04 close: [8.007]


In [20]:
OUTH2 = f"{DIR}/h_prices_hfq.csv"
rows, failed = [], []

for i, hc in enumerate(final.h_code, 1):
    try:
        d = ak.stock_zh_ah_daily(symbol=hc, start_year="2013",
                                 end_year="2020", adjust="hfq")
        d = d.iloc[:, :6]
        d.columns = ["date","open","close","high","low","volume"]
        d["date"] = pd.to_datetime(d.date)
        d = d[(d.date >= "2013-01-01") & (d.date <= "2020-12-31")]
        d = d.drop_duplicates("date")
        d["h_code"] = hc
        rows.append(d)
    except Exception as e:
        failed.append((hc, type(e).__name__))
    if i % 15 == 0:
        print(f"  {i}/{len(final)}  kept {len(rows)}  failed {len(failed)}")
    time.sleep(0.8)

H2 = pd.concat(rows, ignore_index=True).drop_duplicates(["h_code","date"])
H2.to_csv(OUTH2, index=False)
print(f"\nH hfq series: {H2.h_code.nunique()} of {len(final)}")
print(f"rows: {len(H2):,}   {H2.date.min().date()} to {H2.date.max().date()}")
n = H2.groupby("h_code").size()
print(f"days: median {n.median():.0f}, min {n.min()}, max {n.max()}")
if failed: print(f"failed: {failed[:8]}")

  15/96  kept 14  failed 1


  30/96  kept 28  failed 2


  45/96  kept 42  failed 3


  60/96  kept 56  failed 4


  75/96  kept 71  failed 4


  90/96  kept 86  failed 4



H hfq series: 91 of 96
rows: 155,589   2013-01-02 to 2020-12-31
days: median 1969, min 100, max 1969
failed: [('06160', 'KeyError'), ('09995', 'KeyError'), ('01877', 'KeyError'), ('09969', 'KeyError'), ('01108', 'KeyError')]


In [21]:
h2 = H2[H2.h_code == "00939"].set_index("date").close
a  = Ap[Ap.a_code == "601939"].set_index("date").close

r = pd.DataFrame({"h": h2, "a": a}).dropna()
r["ratio"] = r.h / r.a
print(f"CCB overlapping days: {len(r)}")
print(f"ratio: start {r.ratio.iloc[0]:.3f}  end {r.ratio.iloc[-1]:.3f}"
      f"  mean {r.ratio.mean():.3f}  sd {r.ratio.std():.3f}")

big = r.ratio.pct_change().abs().nlargest(8)
print("\n8 largest one-day ratio moves (was up to 10% before):")
print((100*big).round(1).to_string())

# the pair sample: firms with both sides
pairs = set(H2.h_code) & set(final[final.a_code.isin(Ap.a_code)].h_code)
print(f"\nfirms with BOTH sides: {len(pairs)}")
sub = final[final.h_code.isin(pairs)]
print(f"  Shanghai: {sub.a_code.str.startswith('6').sum()}")
print(f"  Shenzhen: {(~sub.a_code.str.startswith('6')).sum()}")

CCB overlapping days: 1888
ratio: start 1.385  end 0.888  mean 1.043  sd 0.199

8 largest one-day ratio moves (was up to 10% before):
date
2015-07-07    9.6
2015-07-06    9.4
2015-09-01    9.3
2015-01-19    7.7
2013-07-02    7.4
2015-08-24    7.2
2014-12-05    7.0
2015-09-07    6.6

firms with BOTH sides: 83
  Shanghai: 64
  Shenzhen: 19


In [22]:
P = (H2.rename(columns={"close":"h_close"})[["date","h_code","h_close"]]
       .merge(final[["h_code","a_code"]], on="h_code")
       .merge(Ap.rename(columns={"close":"a_close"})[["date","a_code","a_close"]],
              on=["date","a_code"]))
P["ratio"] = P.h_close / P.a_close
P = P.sort_values(["h_code","date"])
P["dratio"] = P.groupby("h_code").ratio.pct_change()

daily = P.groupby("date").dratio.agg(["mean","std","count"])
top = daily["mean"].abs().nlargest(10)
print("dates with the largest AVERAGE ratio move across all firms:")
print(daily.loc[top.index].round(4).to_string())

print(f"\npanel: {P.h_code.nunique()} firms, {len(P):,} firm-days")
w = P[P.date.dt.month.isin([12,1,2])]
print(f"winter firm-days: {len(w):,}")

dates with the largest AVERAGE ratio move across all firms:
              mean     std  count
date                             
2015-08-25  0.0926  0.0322     51
2015-07-02  0.0877  0.0699     54
2015-07-06 -0.0767  0.0625     54
2015-06-19  0.0622  0.0286     55
2015-06-26  0.0588  0.0291     55
2015-04-08  0.0578  0.0716     53
2015-08-28 -0.0567  0.0621     52
2015-08-18  0.0495  0.0378     52
2014-12-29 -0.0475  0.0432     52
2016-02-25  0.0453  0.0236     54

panel: 83 firms, 117,259 firm-days
winter firm-days: 27,968


In [23]:
P.to_csv(f"{DIR}/ah_panel.csv", index=False)
print("saved ah_panel.csv:", len(P), "firm-days")

saved ah_panel.csv: 117259 firm-days


In [24]:
CN.index.name = "date"
sh = CN["Shanghai"].rename("pm_sh")
sz = CN["Shenzhen"].rename("pm_sz")
hk = hk.rename("pm_hk")          # from the earlier HK build

pol = pd.concat([sh, sz, hk], axis=1)
pol.index = pd.to_datetime(pol.index)

P["date"] = pd.to_datetime(P.date)
P["mkt"] = np.where(P.a_code.str.startswith("6"), "sh", "sz")

M = P.merge(pol, left_on="date", right_index=True, how="inner")
M = M[M.date.dt.month.isin([12, 1, 2])]

M["pm_cn"] = np.where(M.mkt == "sh", M.pm_sh, M.pm_sz)
M = M.dropna(subset=["pm_cn", "pm_hk", "dratio"])

print(f"ESTIMATION SAMPLE")
print(f"  firm-days: {len(M):,}")
print(f"  firms: {M.h_code.nunique()}   trading days: {M.date.nunique()}")
print(f"  Shanghai firms: {M[M.mkt=='sh'].h_code.nunique()}"
      f"   days: {M[M.mkt=='sh'].date.nunique()}")
print(f"  Shenzhen firms: {M[M.mkt=='sz'].h_code.nunique()}"
      f"   days: {M[M.mkt=='sz'].date.nunique()}")
print(f"\n  span: {M.date.min().date()} to {M.date.max().date()}")
print(f"  firm-days per winter:")
print(M.groupby(M.date.dt.year).size().to_string())

NameError: name 'hk' is not defined

In [25]:
GENERAL = ["SHATIN","TSUEN WAN","EASTERN","KWUN TONG","TUEN MUN","TUNG CHUNG",
           "SHAM SHUI PO","YUEN LONG","CENTRAL/WESTERN","KWAI CHUNG",
           "TAP MUN","TSEUNG KWAN O","TAI PO"]

parts = []
for f in sorted(glob.glob(f"{DIR}/hk_pm25_*.csv")):
    d = pd.read_csv(f, skiprows=11, na_values=["N.A."], encoding="utf-8-sig")
    d.columns = [c.strip() for c in d.columns]
    parts.append(d)

Hk = pd.concat(parts, ignore_index=True)
Hk["ts"] = (pd.to_datetime(Hk.DATE, format="%d/%m/%Y")
            + pd.to_timedelta(Hk.HOUR.astype(int) - 1, unit="h"))
Hk = Hk.drop_duplicates("ts").set_index("ts").sort_index()

gen = Hk[GENERAL].apply(pd.to_numeric, errors="coerce")
hk = gen.mean(axis=1).where(gen.notna().sum(axis=1) >= 8).resample("D").mean().dropna()
hk.name = "pm_hk"
print(f"HK rebuilt: {len(hk)} days, mean {hk.mean():.1f}")

HK rebuilt: 4838 days, mean 19.6


In [26]:
CN.index.name = "date"
sh = CN["Shanghai"].rename("pm_sh")
sz = CN["Shenzhen"].rename("pm_sz")
hk = hk.rename("pm_hk")          # from the earlier HK build

pol = pd.concat([sh, sz, hk], axis=1)
pol.index = pd.to_datetime(pol.index)

P["date"] = pd.to_datetime(P.date)
P["mkt"] = np.where(P.a_code.str.startswith("6"), "sh", "sz")

M = P.merge(pol, left_on="date", right_index=True, how="inner")
M = M[M.date.dt.month.isin([12, 1, 2])]

M["pm_cn"] = np.where(M.mkt == "sh", M.pm_sh, M.pm_sz)
M = M.dropna(subset=["pm_cn", "pm_hk", "dratio"])

print(f"ESTIMATION SAMPLE")
print(f"  firm-days: {len(M):,}")
print(f"  firms: {M.h_code.nunique()}   trading days: {M.date.nunique()}")
print(f"  Shanghai firms: {M[M.mkt=='sh'].h_code.nunique()}"
      f"   days: {M[M.mkt=='sh'].date.nunique()}")
print(f"  Shenzhen firms: {M[M.mkt=='sz'].h_code.nunique()}"
      f"   days: {M[M.mkt=='sz'].date.nunique()}")
print(f"\n  span: {M.date.min().date()} to {M.date.max().date()}")
print(f"  firm-days per winter:")
print(M.groupby(M.date.dt.year).size().to_string())

ESTIMATION SAMPLE
  firm-days: 23,968
  firms: 77   trading days: 390
  Shanghai firms: 60   days: 390
  Shenzhen firms: 17   days: 390

  span: 2013-12-02 to 2020-02-28
  firm-days per winter:
date
2013    1078
2014    3028
2015    3053
2016    3168
2017    3298
2018    3418
2019    4153
2020    2772


In [27]:
M.to_csv(f"{DIR}/estimation_sample.csv", index=False)
print("saved:", len(M), "firm-days")

saved: 23968 firm-days


In [2]:
import cdsapi, os
try:
    c = cdsapi.Client(retry_max=1)
    c.retrieve("reanalysis-era5-single-levels", {
        "product_type": "reanalysis", "variable": ["2m_temperature"],
        "year": "2018", "month": "12", "day": "01", "time": "12:00",
        "area": [31.50, 121.25, 31.00, 121.75], "data_format": "netcdf",
    }, "/content/test.nc")
    print("CDS ALIVE —", os.path.getsize("/content/test.nc"), "bytes")
except Exception as e:
    print("still down:", type(e).__name__, str(e)[:100])

ModuleNotFoundError: No module named 'cdsapi'

In [3]:
!pip install cdsapi -q

CDS_KEY = ""     # paste key, run, then clear again
with open("/root/.cdsapirc", "w") as f:
    f.write("url: https://cds.climate.copernicus.eu/api\n")
    f.write(f"key: {CDS_KEY}\n")


In [4]:
import cdsapi, os
try:
    c = cdsapi.Client(retry_max=1)
    c.retrieve("reanalysis-era5-single-levels", {
        "product_type": "reanalysis", "variable": ["2m_temperature"],
        "year": "2018", "month": "12", "day": "01", "time": "12:00",
        "area": [31.50, 121.25, 31.00, 121.75], "data_format": "netcdf",
    }, "/content/test.nc")
    print("CDS ALIVE —", os.path.getsize("/content/test.nc"), "bytes")
except Exception as e:
    print("still down:", type(e).__name__, str(e)[:100])

still down: HTTPError 500 Server Error: Internal Server Error for url: https://cds.climate.copernicus.eu/api/retrieve/v1/p


In [5]:
import requests, pandas as pd

def om(lat, lon, name):
    r = requests.get("https://archive-api.open-meteo.com/v1/archive", params={
        "latitude": lat, "longitude": lon,
        "start_date": "2018-12-01", "end_date": "2018-12-31",
        "hourly": ",".join(["boundary_layer_height","wind_speed_10m",
                            "wind_direction_10m","surface_pressure",
                            "temperature_2m","precipitation",
                            "cloud_cover","shortwave_radiation"]),
        "timezone": "UTC"}, timeout=60)
    print(f"\n=== {name} === HTTP {r.status_code}")
    if r.status_code != 200:
        print(r.text[:300]); return None
    j = r.json()
    d = pd.DataFrame(j["hourly"])
    print("variables:", [c for c in d.columns if c != "time"])
    print(f"hours: {len(d)}")
    blh = d.get("boundary_layer_height")
    if blh is not None:
        print(f"BLH: mean {blh.mean():.0f}  min {blh.min():.0f}  max {blh.max():.0f}"
              f"  missing {blh.isna().sum()}")
    return d

sh = om(31.25, 121.50, "Shanghai")
hk = om(22.25, 114.25, "Hong Kong")


=== Shanghai === HTTP 200
variables: ['boundary_layer_height', 'wind_speed_10m', 'wind_direction_10m', 'surface_pressure', 'temperature_2m', 'precipitation', 'cloud_cover', 'shortwave_radiation']
hours: 744
BLH: mean 556  min 20  max 1420  missing 0

=== Hong Kong === HTTP 200
variables: ['boundary_layer_height', 'wind_speed_10m', 'wind_direction_10m', 'surface_pressure', 'temperature_2m', 'precipitation', 'cloud_cover', 'shortwave_radiation']
hours: 744
BLH: mean 355  min 15  max 1105  missing 0


In [6]:
import numpy as np, time

def pull_city(lat, lon, tag):
    frames = []
    for y0, y1 in [(2013,2014),(2014,2015),(2015,2016),(2016,2017),
                   (2017,2018),(2018,2019),(2019,2020)]:
        r = requests.get("https://archive-api.open-meteo.com/v1/archive", params={
            "latitude": lat, "longitude": lon,
            "start_date": f"{y0}-12-01", "end_date": f"{y1}-02-29",
            "hourly": ",".join(["boundary_layer_height","wind_speed_10m",
                                "wind_direction_10m","surface_pressure",
                                "temperature_2m","precipitation",
                                "cloud_cover","shortwave_radiation"]),
            "timezone": "UTC"}, timeout=90)
        d = pd.DataFrame(r.json()["hourly"])
        frames.append(d)
        print(f"  {tag} {y0}-{y1}: {len(d)} hours")
        time.sleep(1)

    M = pd.concat(frames, ignore_index=True).drop_duplicates("time")
    M["time"] = pd.to_datetime(M.time, utc=True)

    # wind speed/direction -> u,v  (direction is where the wind comes FROM)
    th = np.radians(M.wind_direction_10m)
    M["u"] = -M.wind_speed_10m * np.sin(th)
    M["v"] = -M.wind_speed_10m * np.cos(th)
    M["city"] = tag
    return M

SH = pull_city(31.25, 121.50, "shanghai")
HK = pull_city(22.25, 114.25, "hongkong")

MET = pd.concat([SH, HK], ignore_index=True)
MET.to_csv(f"{DIR}/met_hourly.csv", index=False)
print(f"\nsaved {len(MET):,} city-hours")
print(f"span {MET.time.min().date()} to {MET.time.max().date()}")
print(MET.groupby("city").size().to_string())

  shanghai 2013-2014: 2184 hours
  shanghai 2014-2015: 2184 hours
  shanghai 2015-2016: 2184 hours
  shanghai 2016-2017: 2184 hours
  shanghai 2017-2018: 2184 hours
  shanghai 2018-2019: 2184 hours
  shanghai 2019-2020: 2184 hours
  hongkong 2013-2014: 2184 hours
  hongkong 2014-2015: 2184 hours
  hongkong 2015-2016: 2184 hours
  hongkong 2016-2017: 2184 hours
  hongkong 2017-2018: 2184 hours
  hongkong 2018-2019: 2184 hours
  hongkong 2019-2020: 2184 hours


NameError: name 'DIR' is not defined

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DIR = "/content/drive/MyDrive/pollution-ah"

MET.to_csv(f"{DIR}/met_hourly.csv", index=False)
print(f"saved {len(MET):,} city-hours")
print(f"span {MET.time.min().date()} to {MET.time.max().date()}")
print(MET.groupby("city").size().to_string())

ValueError: mount failed